# 01 数值积分问题概览

数值积分的目标是用有限次函数计算或有限个采样值近似

$$
I(f)=\int_a^b f(x)\,dx.
$$

在数学分析课程中，我们常常先寻找原函数；但在科学计算中，被积函数可能没有初等原函数，可能来自仿真程序，可能只给出离散观测数据，也可能定义在高维区域上。本节先建立数值求积的基本语言：节点、权重、代数精度、误差、收敛阶、稳定性和函数计算次数。

本章和前两章有两条直接联系：第二章的插值多项式是 Newton-Cotes 等插值型求积公式的来源；第三章的正交多项式是 Gauss 型求积公式的来源。


## 函数积分、离散数据积分和高维积分

**函数型积分**假设可以按需计算 $f(x)$。此时算法可以主动选择节点，例如等距节点、Gauss 节点或自适应节点。

**离散数据积分**只给出

$$
(x_0,y_0),(x_1,y_1),\ldots,(x_n,y_n),
$$

不能随意增加函数值。算法通常先在相邻节点间做局部插值，再对插值函数积分。

**多重积分和高维积分**面对区域和维数问题。张量积求积在二维、三维中很自然，但如果每个方向使用 $m$ 个节点，$d$ 维中会产生 $m^d$ 个函数值，这就是维数灾难的一个表现。


In [ ]:
import math
import pathlib
import sys

import matplotlib.pyplot as plt
import numpy as np

plt.rcParams["font.sans-serif"] = [
    "Arial Unicode MS",
    "PingFang SC",
    "Heiti SC",
    "SimHei",
    "Noto Sans CJK SC",
]
plt.rcParams["axes.unicode_minus"] = False

ROOT = pathlib.Path.cwd().resolve()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "pyproject.toml").exists() and (candidate / "src" / "py_sc").exists():
        ROOT = candidate
        break
SRC = ROOT / "src"
CHAPTER = ROOT / "chapters" / "ch04_numerical_integration"
SCRIPTS = CHAPTER / "scripts"
for path in [SRC, SCRIPTS]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from py_sc import (
    closed_newton_cotes_weights,
    composite_simpson,
    composite_trapezoid,
    gauss_legendre_integrate,
)


## 求积节点和权重

一般求积公式写成

$$
Q(f)=\sum_{i=0}^n w_i f(x_i),
$$

其中 $x_i$ 是节点，$w_i$ 是权重。节点决定在哪里计算函数，权重决定这些函数值如何组合。

如果从插值出发，令 $\ell_i(x)$ 是 Lagrange 基函数，则

$$
f(x)\approx p_n(x)=\sum_{i=0}^n f(x_i)\ell_i(x).
$$

两边积分得到

$$
\int_a^b f(x)\,dx\approx
\sum_{i=0}^n f(x_i)\int_a^b \ell_i(x)\,dx.
$$

因此插值型求积的权重就是 Lagrange 基函数的积分。


In [ ]:
def teaching_interpolatory_weights(nodes, a=0.0, b=1.0):
    """教学式权重构造：让公式精确积分 1, x, ..., x^n。"""
    nodes = np.asarray(nodes, dtype=float)
    degree = nodes.size - 1
    vandermonde = np.vander(nodes, degree + 1, increasing=True).T
    powers = np.arange(degree + 1, dtype=float)
    moments = (b ** (powers + 1) - a ** (powers + 1)) / (powers + 1)
    return np.linalg.solve(vandermonde, moments)

nodes = np.array([0.0, 0.5, 1.0])
weights = teaching_interpolatory_weights(nodes)
print("Simpson 标准节点:", nodes)
print("Simpson 标准权重:", weights)
print("权重和:", weights.sum())


## 代数精度

如果一个求积公式能对所有次数不超过 $m$ 的多项式精确成立，就说它的代数精度至少为 $m$。

检查代数精度时，只需要测试单项式 $1,x,x^2,\ldots$。例如标准区间 $[0,1]$ 上，精确积分为

$$
\int_0^1 x^k\,dx=\frac{1}{k+1}.
$$

插值型公式使用 $n+1$ 个节点时通常至少有 $n$ 次代数精度；如果节点对称，精度可能额外提高。Simpson 公式使用三个节点，却能精确积分三次多项式。


In [ ]:
def algebraic_degree(nodes, weights, max_degree=10, tol=1e-12):
    nodes = np.asarray(nodes, dtype=float)
    weights = np.asarray(weights, dtype=float)
    degree = -1
    for k in range(max_degree + 1):
        numerical = np.dot(weights, nodes**k)
        exact = 1.0 / (k + 1)
        if abs(numerical - exact) < tol:
            degree = k
        else:
            break
    return degree

trap_nodes = np.array([0.0, 1.0])
trap_weights = teaching_interpolatory_weights(trap_nodes)
simp_nodes = np.array([0.0, 0.5, 1.0])
simp_weights = teaching_interpolatory_weights(simp_nodes)

print("梯形公式代数精度:", algebraic_degree(trap_nodes, trap_weights))
print("Simpson 公式代数精度:", algebraic_degree(simp_nodes, simp_weights))


## 误差、收敛阶和函数计算次数

求积误差是

$$
E(f)=I(f)-Q(f).
$$

如果步长 $h$ 缩小时误差满足

$$
|E_h|\approx C h^p,
$$

则 $p$ 是收敛阶。实际实验中常用步长减半估计

$$
p\approx \frac{\log(e_h/e_{h/2})}{\log 2}.
$$

只比较误差是不够的，还要比较函数计算次数。高阶公式如果需要更多函数值，未必在同一成本下更好。


In [ ]:
def estimate_order(errors):
    errors = np.asarray(errors, dtype=float)
    return np.log(errors[:-1] / errors[1:]) / np.log(2.0)

exact = math.e - 1.0
subintervals = np.array([4, 8, 16, 32, 64])
trap_errors = np.array([abs(composite_trapezoid(np.exp, 0.0, 1.0, int(n)) - exact) for n in subintervals])
simp_errors = np.array([abs(composite_simpson(np.exp, 0.0, 1.0, int(n if n % 2 == 0 else n + 1)) - exact) for n in subintervals])

print("n   梯形误差       Simpson 误差")
for n, et, es in zip(subintervals, trap_errors, simp_errors):
    print(f"{n:<3d} {et: .3e}    {es: .3e}")
print("梯形实验阶:", estimate_order(trap_errors))
print("Simpson 实验阶:", estimate_order(simp_errors))

plt.figure(figsize=(7, 4.2))
plt.loglog(subintervals, trap_errors, "o-", label="复合梯形")
plt.loglog(subintervals, simp_errors, "s-", label="复合 Simpson")
plt.xlabel("子区间数 n")
plt.ylabel("绝对误差")
plt.title("误差随区间划分加密而下降")
plt.grid(True, which="both", alpha=0.3)
plt.legend()
plt.show()


## 方法适用条件和失败情形

* 低阶复合公式适合光滑性一般、需要稳健计算的场景。
* 高阶插值型公式对光滑函数可能很高效，但高阶 Newton-Cotes 权重可能出现较大的正负交替，容易放大扰动。
* 自适应方法适合局部尖峰或局部快速变化的函数，但需要可靠的局部误差估计。
* Gauss 型求积适合光滑函数和带权积分，但节点通常不等距，不适合直接处理固定实验数据。
* 高维问题中，张量积确定性公式的函数计算次数增长很快，Monte Carlo 方法常成为更现实的选择。

本节建立术语。后续 Notebook 会分别展开 Newton-Cotes、复合公式、Romberg、自适应积分、高斯求积、离散数据积分、多重积分和 Monte Carlo 积分。
